# 🧹 Day 2: Data Cleaning & SQL Database Design
**Bluestock Fintech | Mutual Fund Analytics Capstone 2026**

Covers: NAV ffill · transaction cleaning · schema design · SQLite load · 10 SQL queries

## 2.1 Setup

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path().resolve().parent
RAW_DIR  = BASE_DIR / 'data' / 'raw'
PROC_DIR = BASE_DIR / 'data' / 'processed'
DB_PATH  = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'
print(f"DB: {DB_PATH} | exists: {DB_PATH.exists()}")

## 2.2 NAV Cleaning — Forward-Fill Missing Dates

NAV only exists for business days. We forward-fill weekends/holidays so time-series analysis is continuous.

In [ ]:
nav = pd.read_csv(RAW_DIR/'02_nav_history.csv', parse_dates=['date'])
print(f"Before: {nav.shape} | Dupes: {nav.duplicated(['amfi_code','date']).sum()} | NAV<=0: {(nav['nav']<=0).sum()}")

nav.drop_duplicates(subset=['amfi_code','date'], inplace=True)
nav = nav[nav['nav'] > 0].copy()

pivot    = nav.pivot(index='date', columns='amfi_code', values='nav')
full_idx = pd.date_range(pivot.index.min(), pivot.index.max(), freq='B')
pivot    = pivot.reindex(full_idx).ffill()

nav_clean = (pivot.reset_index()
               .melt(id_vars='index', var_name='amfi_code', value_name='nav')
               .rename(columns={'index':'date'})
               .dropna(subset=['nav'])
               .sort_values(['amfi_code','date'])
               .reset_index(drop=True))
nav_clean['daily_return_pct'] = nav_clean.groupby('amfi_code')['nav'].pct_change().round(6)
print(f"After : {nav_clean.shape}")
print(nav_clean[nav_clean['amfi_code']==119551].tail(4)[['date','nav','daily_return_pct']].to_string(index=False))

## 2.3 Investor Transactions — Cleaning

In [ ]:
tx = pd.read_csv(RAW_DIR/'08_investor_transactions.csv', parse_dates=['transaction_date'])
print(f"Before: {tx.shape}")
tx = tx[tx['amount_inr'] > 0].copy()
tx.drop_duplicates(inplace=True)
tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()
tx['city_tier']        = tx['city_tier'].str.upper().str.strip()
print(f"After : {tx.shape}")
print("Transaction types:", tx['transaction_type'].value_counts().to_dict())
print("KYC status       :", tx['kyc_status'].value_counts().to_dict())

## 2.4 Scheme Performance — Validation

In [ ]:
perf = pd.read_csv(RAW_DIR/'07_scheme_performance.csv')
for col in ['return_1yr_pct','return_3yr_pct','sharpe_ratio','alpha','beta','max_drawdown_pct']:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')
print("Scheme Performance — key stats:")
print(perf[['return_3yr_pct','sharpe_ratio','alpha','beta','max_drawdown_pct']].describe().round(3).to_string())
print(f"\nNegative Sharpe : {(perf['sharpe_ratio']<0).sum()}")
print(f"Expense range   : {perf['expense_ratio_pct'].min():.2f}% – {perf['expense_ratio_pct'].max():.2f}%")

## 2.5 Database Schema — Verify Star Schema

In [ ]:
conn = sqlite3.connect(DB_PATH)
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print(f"{'Table':<30} {'Type':<6} {'Rows':>8}")
print("─"*46)
for (t,) in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0]
    ttype = "DIM" if t.startswith('dim') else "FACT"
    print(f"  {t:<28} {ttype:<6} {count:>8,}")
conn.close()

## 2.6 Run 10 Analytical SQL Queries

In [ ]:
conn = sqlite3.connect(DB_PATH)

QUERIES = {
"Q1 — Top 5 Funds by AUM":
    "SELECT scheme_name, fund_house, aum_crore FROM fact_performance ORDER BY aum_crore DESC LIMIT 5",
"Q2 — Average NAV by Month (SBI Bluechip)":
    "SELECT strftime('%Y-%m',date) AS month, ROUND(AVG(nav),2) avg_nav FROM fact_nav WHERE amfi_code=119551 GROUP BY month ORDER BY month DESC LIMIT 6",
"Q3 — SIP Inflow YoY Growth (latest 6)":
    "SELECT month, sip_inflow_crore, ROUND(yoy_growth_pct,1) yoy_pct FROM fact_sip_industry WHERE yoy_growth_pct IS NOT NULL ORDER BY month DESC LIMIT 6",
"Q4 — SIP Amount by State":
    "SELECT state, COUNT(*) txns, ROUND(SUM(amount_inr)/1e7,1) total_cr FROM fact_transactions WHERE transaction_type='Sip' GROUP BY state ORDER BY total_cr DESC",
"Q5 — Funds Expense Ratio < 1% (Direct)":
    "SELECT scheme_name, expense_ratio_pct FROM dim_fund WHERE expense_ratio_pct < 1.0 AND plan='Direct' ORDER BY expense_ratio_pct",
"Q6 — Sharpe Ratio > 1.0 (Outperformers)":
    "SELECT scheme_name, category, sharpe_ratio, sortino_ratio FROM fact_performance WHERE sharpe_ratio > 1.0 ORDER BY sharpe_ratio DESC",
"Q7 — SIP vs Lumpsum by Age Group":
    "SELECT age_group, transaction_type, COUNT(*) n, ROUND(AVG(amount_inr),0) avg_amt FROM fact_transactions GROUP BY age_group, transaction_type ORDER BY age_group",
"Q8 — Top Stocks by Fund Count":
    "SELECT stock_name, sector, COUNT(DISTINCT amfi_code) funds, ROUND(AVG(weight_pct),2) avg_wt FROM fact_portfolio GROUP BY stock_name ORDER BY funds DESC LIMIT 10",
"Q9 — Folio Count Growth":
    "SELECT month, total_folios_crore, equity_folios_crore FROM fact_folio_count ORDER BY month",
"Q10 — Fund Alpha Ranking":
    "SELECT scheme_name, ROUND(return_3yr_pct,2) ret_3yr, ROUND(alpha,2) alpha, CASE WHEN alpha>2 THEN 'Strong Outperformer' WHEN alpha>0 THEN 'Outperformer' ELSE 'Underperformer' END label FROM fact_performance ORDER BY alpha DESC",
}

for title, q in QUERIES.items():
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print("─"*55)
    print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

---
## ✅ Day 2 Complete
- 10 datasets cleaned, NAV forward-filled
- SQLite DB: **11 tables, 88,000+ rows**
- `schema.sql` + `queries.sql` + `data_dictionary.md` generated